In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

import config
from etl.data_loader import DataLoader

loader = DataLoader()

print("DATA_ROOT:", config.PathConfig.DATA_ROOT)
print("PROCESSED:", config.PathConfig.PROCESSED)
print("RAW:", config.PathConfig.RAW)

DATA_ROOT: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage
PROCESSED: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage\processed
RAW: D:\work and study\PostGraduate\HK\project\TradingSystem\data_storage\raw


In [15]:
processed_files = sorted(Path(config.PathConfig.PROCESSED).glob("*.parquet"))

pd.DataFrame({
    "file": [p.name for p in processed_files],
    "path": [str(p) for p in processed_files],
    "size_mb": [round(p.stat().st_size / 1024 / 1024, 2) for p in processed_files],
})

,file,path,size_mb
0,BNBUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.09
1,BNBUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.20
2,BNBUSDT_1m.parquet,D:\work and study\PostGraduate\HK\project\Trad...,75.01
3,BNBUSDT_4h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.55
4,BTCUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.10
5,BTCUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.65
6,BTCUSDT_1m.parquet,D:\work and study\PostGraduate\HK\project\Trad...,90.24
7,BTCUSDT_4h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.61
8,ETHUSDT_1d.parquet,D:\work and study\PostGraduate\HK\project\Trad...,0.10
9,ETHUSDT_1h.parquet,D:\work and study\PostGraduate\HK\project\Trad...,2.52


In [3]:
symbol = "BTC/USDT"
timeframe = "4h"

df = loader.get_crypto_kline_data(
    symbol=symbol,
    timeframe=timeframe,
)

df.head()

,high,net_taker_vol,volume,low,taker_buy_vol,open,close
timestamp,,,,,,,
2021-01-01 00:00:00,29546.42,495.037,43210.161,28706.00,21852.599,28948.19,29302.11
2021-01-01 04:00:00,29422.32,-3136.344,26682.086,28822.00,11772.871,29302.11,29107.71
2021-01-01 08:00:00,29454.45,-339.046,29562.630,28900.00,14611.792,29107.72,29341.99
2021-01-01 12:00:00,29668.86,-1227.234,49142.952,29043.75,23957.859,29342.00,29210.84
2021-01-01 16:00:00,29388.10,-3641.582,43668.170,28627.12,20013.294,29210.85,29048.47


In [17]:
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "missing": [df[c].isna().sum() for c in df.columns],
        "missing_pct": [df[c].isna().mean() for c in df.columns],
        "sample_value": [df[c].dropna().iloc[0] if df[c].dropna().shape[0] else None for c in df.columns],
    })

describe_df(df)

,column,dtype,non_null,missing,missing_pct,sample_value
0,taker_buy_vol,float64,11924,0,0.0,21852.599
1,net_taker_vol,float64,11924,0,0.0,495.037
2,close,float64,11924,0,0.0,29302.110
3,volume,float64,11924,0,0.0,43210.161
4,high,float64,11924,0,0.0,29546.420
5,open,float64,11924,0,0.0,28948.190
6,low,float64,11924,0,0.0,28706.000


In [18]:
pd.DataFrame({
    "symbol": [symbol],
    "timeframe": [timeframe],
    "start": [df.index.min()],
    "end": [df.index.max()],
    "rows": [len(df)],
    "columns": [list(df.columns)],
})

,symbol,timeframe,start,end,rows,columns
0,BTC/USDT,4h,2021-01-01,2026-06-11 04:00:00,11924,"[taker_buy_vol, net_taker_vol, close, volume, ..."


In [24]:
matrix = loader.get_crypto_matrix(
    symbols=["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT"],
    timeframe="4h",
    columns=["close", "volume", "net_taker_vol"],
)

close = matrix["volume"]
close.tail()

读取加密货币数据 (周期: 4h)...
✅ 成功加载 3 个特征矩阵。


,BTC/USDT,ETH/USDT,SOL/USDT,BNB/USDT
timestamp,,,,
2026-06-09 12:00:00,71792.227,1624864.053,7920044.95,168656.41
2026-06-09 16:00:00,51223.379,1356989.394,6688645.28,100301.68
2026-06-09 20:00:00,17544.777,513326.636,2057497.20,51072.06
2026-06-10 00:00:00,19223.281,565567.199,2573453.63,64341.82
2026-06-10 04:00:00,19399.655,539454.290,2311561.68,81668.74


In [6]:
#资金费率表格查询
fund_rate = loader.get_funding_rate_data()
raw_rate = loader.get_raw_funding_rate_data("BTC/USDT")
display(raw_rate)

,symbol,funding_rate,source,created_at
timestamp,,,,
2021-01-01 00:00:00.002,BTC/USDT,0.000228,binance_usdm,2026-06-11 08:24:37.968379
2021-01-01 08:00:00.006,BTC/USDT,0.000263,binance_usdm,2026-06-11 08:24:37.968379
2021-01-01 16:00:00.003,BTC/USDT,0.000345,binance_usdm,2026-06-11 08:24:37.968379
2021-01-02 00:00:00.000,BTC/USDT,0.000100,binance_usdm,2026-06-11 08:24:37.968379
2021-01-02 08:00:00.000,BTC/USDT,0.000202,binance_usdm,2026-06-11 08:24:37.968379
...,...,...,...,...
2026-06-10 00:00:00.001,BTC/USDT,0.000004,binance_usdm,2026-06-11 08:24:37.968379
2026-06-10 08:00:00.011,BTC/USDT,-0.000034,binance_usdm,2026-06-11 08:24:37.968379
2026-06-10 16:00:00.009,BTC/USDT,0.000025,binance_usdm,2026-06-11 08:24:37.968379


In [6]:
from etl.feature_builder_H import build_crypto_features, FeatureBuilderConfig,get_feature_definitions



cfg = FeatureBuilderConfig(
    decision_timeframe="4h",
    include_funding=True,
    include_oi=True,
    include_cvd_proxy=True,
    include_sentiment=True,
    include_onchain=True,
)

# features = build_crypto_features(cfg=cfg, save=True)
# display(features)
defs = get_feature_definitions()
display(defs)

,feature,group,definition,calculation,source,usage
0,symbol,identity,"Trading pair, e.g. BTC/USDT.",Copied from config.TargetConfig.COINS / source...,config / processed market files,Primary entity key. Not a numeric model feature.
1,ts_open,time_grid,Open timestamp of the 4h decision bar.,"processed 4h bar timestamp. With label='left',...",processed/{SYMBOL}_4h.parquet.timestamp,Audit / explainability. Not a model feature by...
2,ts_close,time_grid,Close timestamp of the 4h decision bar.,ts_open + 4h.,derived from ts_open,Defines when the 4h bar has completed.
3,decision_time,time_grid,Time at which the strategy/Agent is allowed to...,ts_close + cfg.market_latency; default = ts_cl...,derived,Primary PIT merge key. All features must satis...
4,open_4h,market_4h,Open price of the completed 4h bar.,first 1m open within the 4h resample window.,processed/{SYMBOL}_4h.parquet.open,4h price structure.
5,high_4h,market_4h,High price of the completed 4h bar.,max 1m high within the 4h resample window.,processed/{SYMBOL}_4h.parquet.high,4h price range / volatility proxy.
6,low_4h,market_4h,Low price of the completed 4h bar.,min 1m low within the 4h resample window.,processed/{SYMBOL}_4h.parquet.low,4h price range / volatility proxy.
7,close_4h,market_4h,Close price of the completed 4h bar.,last 1m close within the 4h resample window.,processed/{SYMBOL}_4h.parquet.close,Main price anchor for returns and downstream b...
8,volume_4h,market_4h,Total traded volume in the completed 4h bar.,sum of 1m volume within the 4h resample window.,processed/{SYMBOL}_4h.parquet.volume,Liquidity / activity state.
9,ret_4h,market_4h,Most recent 4h return.,close_4h.pct_change(1).,derived from close_4h,Short momentum / immediate price change.
